IC50 pipeline. 2 excel files: 1) plate template 2) MTS raw data

In [14]:
import pandas as pd
import numpy as np
from openpyxl.utils.cell import column_index_from_string, get_column_letter

RAW_PATH = "Raw_MTS.xlsx"
TPL_PATH = "Plate_template.xlsx"

In [15]:
# Region of interest in Excel coordinates (easy to change later)
ROI = {
    "col_start": "D",
    "col_end": "M",
    "row_start": 28,
    "row_end": 39
}

In [16]:
def read_block(path, sheet_name=0, col_start="D", col_end="M", row_start=28, row_end=39):
    df = pd.read_excel(path, sheet_name=sheet_name, header=None, engine="openpyxl")

    c0 = column_index_from_string(col_start) - 1  # 0-based
    c1 = column_index_from_string(col_end)        # exclusive

    r0 = row_start - 1
    r1 = row_end

    block = df.iloc[r0:r1, c0:c1].copy()

    # Label columns D..M
    block.columns = [get_column_letter(i) for i in range(column_index_from_string(col_start),
                                                        column_index_from_string(col_end) + 1)]

    expected_rows = row_end - row_start + 1
    got_rows = block.shape[0]

    if got_rows != expected_rows:
        print(
            f"[read_block] WARNING: expected {expected_rows} rows (Excel {row_start}:{row_end}) "
            f"but got {got_rows} rows from pandas. "
            f"df shape={df.shape}, slice rows {r0}:{r1}, cols {c0}:{c1}"
        )

    # Set an index that matches what we actually received
    block.index = list(range(row_start, row_start + got_rows))

    return block


In [17]:
tpl_block = read_block(TPL_PATH, **ROI)
raw_block = read_block(RAW_PATH, **ROI)

tpl_block, raw_block

[read_block] WARNING: expected 12 rows (Excel 28:39) but got 11 rows from pandas. df shape=(38, 13), slice rows 27:39, cols 3:13


(           D           E           F          G        H        I        J  \
 28         C     D_0,001  Gen1S_0,01  Gen1S_0,1  Gen1S_1  Gen1S_2  Gen1S_4   
 29       NaN         NaN         NaN        NaN      NaN      NaN      NaN   
 30         C     D_0,001  Gen1S_0,01  Gen1S_0,1  Gen1S_1  Gen1S_2  Gen1S_4   
 31       NaN         NaN         NaN        NaN      NaN      NaN      NaN   
 32         C     D_0,001  Gen1S_0,01  Gen1S_0,1  Gen1S_1  Gen1S_2  Gen1S_4   
 33       NaN         NaN         NaN        NaN      NaN      NaN      NaN   
 34  Gen1S_20  Gen2S_0,01    Gen2S_01    Gen2S_1  Gen2S_2  Gen2S_4  Gen2S_8   
 35       NaN         NaN         NaN        NaN      NaN      NaN      NaN   
 36  Gen1S_20  Gen2S_0,01    Gen2S_01    Gen2S_1  Gen2S_2  Gen2S_4  Gen2S_8   
 37       NaN         NaN         NaN        NaN      NaN      NaN      NaN   
 38  Gen1S_20  Gen2S_0,01    Gen2S_01    Gen2S_1  Gen2S_2  Gen2S_4  Gen2S_8   
 
            K         L         M  
 28   Gen1S_8 

In [18]:
tpl_id = tpl_block.ffill(axis=0)
tpl_id.head()

,D,E,F,G,H,I,J,K,L,M
28,C,"D_0,001","Gen1S_0,01","Gen1S_0,1",Gen1S_1,Gen1S_2,Gen1S_4,Gen1S_8,Gen1S_10,Gen1S_15
29,C,"D_0,001","Gen1S_0,01","Gen1S_0,1",Gen1S_1,Gen1S_2,Gen1S_4,Gen1S_8,Gen1S_10,Gen1S_15
30,C,"D_0,001","Gen1S_0,01","Gen1S_0,1",Gen1S_1,Gen1S_2,Gen1S_4,Gen1S_8,Gen1S_10,Gen1S_15
31,C,"D_0,001","Gen1S_0,01","Gen1S_0,1",Gen1S_1,Gen1S_2,Gen1S_4,Gen1S_8,Gen1S_10,Gen1S_15
32,C,"D_0,001","Gen1S_0,01","Gen1S_0,1",Gen1S_1,Gen1S_2,Gen1S_4,Gen1S_8,Gen1S_10,Gen1S_15


In [19]:
raw_block.iloc[:4]

,D,E,F,G,H,I,J,K,L,M
28,1.778,1.668,1.845,1.640,1.759,1.644,0.452,0.133,0.134,0.137
29,0.675,0.588,0.666,0.574,0.638,0.575,0.155,0.059,0.060,0.058
30,2.067,1.794,2.289,1.670,1.431,2.122,0.831,0.136,0.149,0.134
31,0.778,0.620,0.812,0.603,0.514,0.700,0.306,0.058,0.070,0.057


In [20]:
# Convert raw block to numeric (safe even if already numeric)
raw_num = raw_block.apply(pd.to_numeric, errors="coerce")

# Sanity check: must be even number of rows
if raw_num.shape[0] % 2 != 0:
    raise ValueError(
        f"Expected an even number of rows for top/bottom pairing, "
        f"got {raw_num.shape[0]}"
    )

top = raw_num.iloc[0::2].reset_index(drop=True)
bottom = raw_num.iloc[1::2].reset_index(drop=True)

MTS = top - bottom
MTS

,D,E,F,G,H,I,J,K,L,M
0,1.103,1.080,1.179,1.066,1.121,1.069,0.297,0.074,0.074,0.079
1,1.289,1.174,1.477,1.067,0.917,1.422,0.525,0.078,0.079,0.077
2,1.482,1.254,1.385,1.024,0.839,1.064,0.684,0.095,0.074,0.077
3,0.080,1.383,1.336,1.322,0.940,0.584,0.076,0.070,0.073,0.078
4,0.076,1.360,1.242,1.261,0.868,0.582,0.075,0.072,0.074,0.079
5,0.077,1.478,1.184,1.224,0.869,0.487,0.076,0.073,0.074,0.085


In [21]:
tpl_id_top = tpl_id.iloc[0::2].reset_index(drop=True)
tpl_id_top.index = MTS.index  # align by pair index

tpl_id_top.head()

,D,E,F,G,H,I,J,K,L,M
0,C,"D_0,001","Gen1S_0,01","Gen1S_0,1",Gen1S_1,Gen1S_2,Gen1S_4,Gen1S_8,Gen1S_10,Gen1S_15
1,C,"D_0,001","Gen1S_0,01","Gen1S_0,1",Gen1S_1,Gen1S_2,Gen1S_4,Gen1S_8,Gen1S_10,Gen1S_15
2,C,"D_0,001","Gen1S_0,01","Gen1S_0,1",Gen1S_1,Gen1S_2,Gen1S_4,Gen1S_8,Gen1S_10,Gen1S_15
3,Gen1S_20,"Gen2S_0,01",Gen2S_01,Gen2S_1,Gen2S_2,Gen2S_4,Gen2S_8,Gen2S_10,Gen2S_15,Gen2S_20
4,Gen1S_20,"Gen2S_0,01",Gen2S_01,Gen2S_1,Gen2S_2,Gen2S_4,Gen2S_8,Gen2S_10,Gen2S_15,Gen2S_20


In [22]:
long = (
    MTS.reset_index(names="pair_row")
       .melt(id_vars="pair_row", var_name="col", value_name="MTS")
       .merge(
           tpl_id_top.reset_index(names="pair_row")
                    .melt(id_vars="pair_row", var_name="col", value_name="Identity"),
           on=["pair_row", "col"],
           how="left"
       )
)

In [23]:
summary = (
    long.groupby("Identity", as_index=False)
        .agg(
            n=("MTS", "count"),
            mts_mean=("MTS", "mean"),
            mts_sd=("MTS", "std")
        )
)

In [24]:
# Check: identity counts should look like 3-ish (or whatever you expect)
summary[["Identity", "n"]].head(30)

,Identity,n
0,C,3
1,"D_0,001",3
2,"Gen1S_0,01",3
3,"Gen1S_0,1",3
4,Gen1S_1,3
5,Gen1S_10,3
6,Gen1S_15,3
7,Gen1S_2,3
8,Gen1S_20,3
9,Gen1S_4,3


In [25]:
# Look at one identity's replicate values
one = summary.loc[summary["n"].idxmax(), "Identity"]
long[long["Identity"] == one].sort_values(["pair_row","col"])

,pair_row,col,MTS,Identity
0,0,D,1.103,C
1,1,D,1.289,C
2,2,D,1.482,C


In [26]:
col_order = list("DEFGHIJKLM")

df = long.copy()

# Ensure proper D..M ordering
df["col"] = pd.Categorical(df["col"], categories=col_order, ordered=True)

# Determine the first 3 and last 3 pair_rows
pair_rows = sorted(df["pair_row"].unique())
if len(pair_rows) < 6:
    raise ValueError(f"Expected at least 6 pair_row values, got {len(pair_rows)}: {pair_rows}")

first3 = set(pair_rows[:3])
last3  = set(pair_rows[3:6])

df["block"] = np.where(df["pair_row"].isin(first3), 1,
               np.where(df["pair_row"].isin(last3), 2, 99))

# Keep only pair_rows 0–5 (first 6 pair rows)
df = df[df["block"] != 99].copy()

# Sort exactly as requested: D(0-2), E(0-2)... M(0-2), then D(3-5)... M(3-5)
export_df = (
    df.sort_values(["block", "col", "pair_row"])
      .loc[:, ["Identity", "MTS", "col", "block"]]
      .rename(columns={"col": "column"})
      .reset_index(drop=True)
)

# Add replicate number (1,2,3) per identity within each block
export_df["Replicate"] = (
    export_df.groupby(["block", "Identity"]).cumcount() + 1
)

# ---- Split Identity into Drug + concentration (nM) + log10(nM) ----
def split_identity_to_nm(identity):
    if pd.isna(identity):
        return pd.Series({
            "Compound": None,
            "Conc_nM": np.nan,
            "log10_Conc_nM": np.nan
        })

    s = str(identity).strip()

    if "_" in s:
        drug, conc = s.split("_", 1)
        conc = conc.replace(",", ".")
        conc_uM = pd.to_numeric(conc, errors="coerce")
        conc_nM = conc_uM * 1000 if pd.notna(conc_uM) else np.nan
        log_nm = np.log10(conc_nM) if pd.notna(conc_nM) and conc_nM > 0 else np.nan

        return pd.Series({
            "Compound": drug,
            "Conc_nM": conc_nM,
            "log10_Conc_nM": log_nm
        })

    # Controls / labels
    return pd.Series({
        "Compound": s,
        "Conc_nM": np.nan,
        "log10_Conc_nM": np.nan
    })

export_df[["Compound", "Conc_nM", "log10_Conc_nM"]] = (
    export_df["Identity"].apply(split_identity_to_nm)
)

# Final: drop internal columns
export_df_final = export_df.drop(columns=["column", "block", "Identity"])

# Final column order (log column immediately after nM)
export_df_final = export_df_final[
    ["Replicate", "Compound", "Conc_nM", "log10_Conc_nM", "MTS"]
]

export_df_final.head(20)

,Replicate,Compound,Conc_nM,log10_Conc_nM,MTS
0,1,C,NaN,NaN,1.103
1,2,C,NaN,NaN,1.289
2,3,C,NaN,NaN,1.482
3,1,D,1.0,0.00000,1.080
4,2,D,1.0,0.00000,1.174
5,3,D,1.0,0.00000,1.254
6,1,Gen1S,10.0,1.00000,1.179
7,2,Gen1S,10.0,1.00000,1.477
8,3,Gen1S,10.0,1.00000,1.385
9,1,Gen1S,100.0,2.00000,1.066


Now were ready for IC50 plotting as all the required information is extracted. 